# Phase 1 Starter — Train & Deploy (Individual, 35%)
This is Session 2's demo generalized: point it at **your** dataset and work top to bottom.
Milestones: S1 pick dataset → S2 first training run → S3 tuned + endpoint attempt →
**working endpoint by start of S4** → report (template in `project/`) due S5.

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [6]:
import sys, pathlib
REPO = pathlib.Path(r"e:\AUBG\5th semester\ML\ml-project")
sys.path.insert(0, str(REPO / 'src'))
import warnings; warnings.filterwarnings('ignore')

from course_utils import data
DATASET = 'credit_default'        # <- your pick from project/datasets.md
TARGET = 'default payment next month'                    # <- your label column
df = data.load_raw(DATASET)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 1. Profile & clean (Session 1 toolkit) — document ≥2 decisions for the report

In [3]:
data.profile_frame(df)
# your cleaning here. Watch for target leakage: is any feature only known AFTER the outcome?

shape: 30,000 rows x 25 cols | duplicated rows: 0


,dtype,missing,missing_%,unique,sample
ID,int64,0,0.0,30000,1
LIMIT_BAL,int64,0,0.0,81,20000
SEX,int64,0,0.0,2,2
EDUCATION,int64,0,0.0,7,2
MARRIAGE,int64,0,0.0,4,1
AGE,int64,0,0.0,56,24
PAY_0,int64,0,0.0,11,2
PAY_2,int64,0,0.0,11,2
PAY_3,int64,0,0.0,11,-1
PAY_4,int64,0,0.0,11,-1


In [4]:
display(df.describe().T[['min', 'max', 'mean']])

,min,max,mean
ID,1.0,30000.0,15000.500000
LIMIT_BAL,10000.0,1000000.0,167484.322667
SEX,1.0,2.0,1.603733
EDUCATION,0.0,6.0,1.853133
MARRIAGE,0.0,3.0,1.551867
AGE,21.0,79.0,35.485500
PAY_0,-2.0,8.0,-0.016700
PAY_2,-2.0,8.0,-0.133767
PAY_3,-2.0,8.0,-0.166200
PAY_4,-2.0,8.0,-0.220667


In [9]:
print('Duplicate rows:', df.duplicated().sum())

Duplicate rows: 0


In [9]:
print('Duplicate rows (without ID):', df.drop(columns=['ID']).duplicated().sum())
dups = df[df.drop(columns=['ID']).duplicated(keep=False)]
display(dups.sort_values(['LIMIT_BAL', 'AGE']))

Duplicate rows (without ID): 35


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
12430,12431,20000,1,2,2,24,2,2,4,4,...,1650,1650,1650,0,0,0,0,0,0,1
14294,14295,20000,1,2,2,24,2,2,4,4,...,1650,1650,1650,0,0,0,0,0,0,1
7170,7171,50000,2,1,2,23,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
17032,17033,50000,2,1,2,23,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
1759,1760,50000,1,2,2,26,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25608,25609,360000,2,1,1,41,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
18,19,360000,2,1,1,49,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
13106,13107,360000,2,1,1,49,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
839,840,500000,1,1,1,43,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
display(df.groupby('PAY_0')[TARGET].mean().round(2))
#Exploratory analysis:

PAY_0
-2    0.13
-1    0.17
 0    0.13
 1    0.34
 2    0.69
 3    0.76
 4    0.68
 5    0.50
 6    0.55
 7    0.78
 8    0.58
Name: default payment next month, dtype: float64

In [ ]:
#Reconstruct
df = df.drop(columns=['ID'])
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
df['EDUCATION'] = df['EDUCATION'].map({1: 'graduate', 2: 'university', 3: 'high_school', 4: 'other'})
df['MARRIAGE'] = df['MARRIAGE'].map({1: 'married', 2: 'single', 3: 'other'})
df = df.drop(columns=['SEX'])


## 2. Features & splits

In [22]:
import pandas as pd
y = (df[TARGET] == 'yes').astype(int) if df[TARGET].dtype == object else df[TARGET]
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)
S = data.make_splits(X, y)
print({k: v.shape for k, v in S.items() if hasattr(v, 'shape')})

{'X_train': (21000, 25), 'y_train': (21000,), 'X_val': (4500, 25), 'y_val': (4500,), 'X_test': (4500, 25), 'y_test': (4500,)}


## 3. Baseline → tuned model (≥2 hyperparameters, show the effect)

In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from course_utils import viz

# 1. Always predict the majority (no default)
dummy = DummyClassifier(strategy='most_frequent').fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], dummy.predict_proba(S['X_val'])[:, 1], name='majority (val)')

# 2. Logistic regression (features scaled first)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], logreg.predict_proba(S['X_val'])[:, 1], name='logistic regression (val)')

# 3. XGBoost
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='XGBoost (val)')

--- majority (val) ---
  accuracy: 0.779
 precision: 0.000
    recall: 0.000
        f1: 0.000
   auc_roc: 0.500
--- logistic regression (val) ---
  accuracy: 0.809
 precision: 0.697
    recall: 0.243
        f1: 0.361
   auc_roc: 0.714
--- XGBoost (val) ---
  accuracy: 0.818
 precision: 0.665
    recall: 0.358
        f1: 0.465
   auc_roc: 0.778


{'accuracy': 0.8182222222222222,
 'precision': 0.6654205607476635,
 'recall': 0.3577889447236181,
 'f1': 0.465359477124183,
 'auc_roc': 0.777998982071556}

| Model | Accuracy | Precision | Recall | F1 | AUC (val) |
|---|---|---|---|---|---|
| Majority baseline | 0.779 | 0.000 | 0.000 | 0.000 | 0.500 |
| Logistic regression | 0.809 | 0.697 | 0.243 | 0.361 | 0.714 |
| XGBoost | 0.818 | 0.665 | 0.358 | 0.465 | 0.778 |

In [24]:
#tuning
from sklearn.metrics import roc_auc_score

def run(**params):
    base = dict(n_estimators=200, max_depth=4, learning_rate=0.1,
                eval_metric='auc', random_state=42)
    base.update(params)
    m = XGBClassifier(**base).fit(S['X_train'], S['y_train'])
    return (roc_auc_score(S['y_train'], m.predict_proba(S['X_train'])[:, 1]),
            roc_auc_score(S['y_val'],   m.predict_proba(S['X_val'])[:, 1]))

# learning_rate stays 0.1
print('--- max_depth ---')
for d in [2, 4, 8, 12]:
    tr, va = run(max_depth=d)
    print(f"max_depth={d}: train AUC={tr:.3f}, val AUC={va:.3f}, gap={tr-va:.3f}")

# max_depth stays 4
print('--- learning_rate ---')
for lr in [0.01, 0.1, 0.3]:
    tr, va = run(learning_rate=lr)
    print(f"learning_rate={lr}: train AUC={tr:.3f}, val AUC={va:.3f}, gap={tr-va:.3f}")

--- max_depth ---
max_depth=2: train AUC=0.800, val AUC=0.778, gap=0.023
max_depth=4: train AUC=0.852, val AUC=0.778, gap=0.074
max_depth=8: train AUC=0.982, val AUC=0.763, gap=0.220
max_depth=12: train AUC=1.000, val AUC=0.752, gap=0.248
--- learning_rate ---
learning_rate=0.01: train AUC=0.791, val AUC=0.771, gap=0.020
learning_rate=0.1: train AUC=0.852, val AUC=0.778, gap=0.074
learning_rate=0.3: train AUC=0.917, val AUC=0.756, gap=0.161


| Hyperparameter | Value | Train AUC | Val AUC | Gap |
|---|---|---|---|---|
| max_depth | 2 | 0.800 | **0.778** | 0.023 |
| max_depth | 4 | 0.852 | 0.778 | 0.074 |
| max_depth | 8 | 0.982 | 0.763 | 0.220 |
| max_depth | 12 | 1.000 | 0.752 | 0.248 |
| learning_rate | 0.01 | 0.791 | 0.771 | 0.020 |
| learning_rate | **0.1** | 0.852 | **0.778** | 0.074 |
| learning_rate | 0.3 | 0.917 | 0.756 | 0.161 |

In [25]:
# Final model
model = XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.1,
                    eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='XGBoost tuned (val)')

--- XGBoost tuned (val) ---
  accuracy: 0.818
 precision: 0.665
    recall: 0.355
        f1: 0.463
   auc_roc: 0.778


{'accuracy': 0.8177777777777778,
 'precision': 0.664783427495292,
 'recall': 0.35477386934673366,
 'f1': 0.46264744429882043,
 'auc_roc': 0.777621058215471}

## 4. Final honest number — test set, touched ONCE

In [8]:
final = viz.metric_report(S['y_test'], model.predict_proba(S['X_test'])[:, 1], name='FINAL (test)')
from course_utils import models
models.save_model(model, f'phase1_{DATASET}', metrics=final)

NameError: name 'viz' is not defined

## 5. Deploy
```python
from course_utils import aws
aws.deploy_sklearn_model(f'models/phase1_{DATASET}.joblib', endpoint_name=f'phase1-{DATASET}-YOURNAME')
# invoke it, screenshot it for the report, then DELETE it:
# aws.delete_endpoint(f'phase1-{DATASET}-YOURNAME')
```
Sandbox blocking you? Local FastAPI twin accepted with prior approval — see `session4/local_api.py`.